In [ ]:
import json
from pathlib import Path
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


In [ ]:
# ====== CONFIG ======

COCO_JSON_PATHS = [
    Path("/content/instances_default.json"),
    Path("/content/instances_Train.json"),
]

# Output CSV clean trung gian (cho debug / train tabular)
OUT_CLEAN_CSV = Path("/content/drive/MyDrive/pig-selected_all-image/behavior_clean_merged.csv")

OUT_COCO_CLEAN = Path("/content/drive/MyDrive/pig-selected_all-image/instances_clean.json")

IMG_ROOTS = [
    Path("/content/drive/MyDrive/pig-selected_frame_attribute_(3)/images_frame_attribute_(3)"),
    Path("/content/drive/MyDrive/pig-selected_frame_attribute_(4)/images_frame_attribute_(4)"),
    Path("/content/drive/MyDrive/pig-selected_frame_attribute/images_frame_attribute"),
]

IMG_CLEAN_ROOT = Path("/content/drive/MyDrive/pig-selected_all-image/images_clean")
IMG_CLEAN_ROOT.mkdir(parents=True, exist_ok=True)

BEHAVIORS = [
    "drink",
    "eat",
    "fight",
    "social-nose",
    "explore",
    "lying",
    "stand",
    "move",
    "sitting",
    "playwithtoy",
]
BEHAVIOR_SET = set(BEHAVIORS)

DROP_HIDDEN = False

print("COCO_JSON_PATHS:", COCO_JSON_PATHS)
print("OUT_CLEAN_CSV  :", OUT_CLEAN_CSV)
print("OUT_COCO_CLEAN :", OUT_COCO_CLEAN)
print("IMG_ROOTS      :", IMG_ROOTS)
print("IMG_CLEAN_ROOT :", IMG_CLEAN_ROOT)


In [ ]:
def parse_burst_from_filename(img_name: str):
    """
      burst_{vid_stem}_{vid_hash}_{ms}_f{fi}_k{order}.jpg
    """
    stem = Path(img_name).stem
    parts = stem.split("_")
    if len(parts) < 4:
        return stem, 0

    k_part = parts[-1]  # 'k3'
    try:
        order = int(k_part.replace("k", ""))
    except:
        order = 0

    group_id = "_".join(parts[:-2])  # 'burst_{vid_stem}_{hash}_{ms}'
    return group_id, order


In [ ]:
def load_and_merge_coco_with_burst(json_paths, pig_category_name="Pig") -> pd.DataFrame:
    """
      img_name, width, height,
      x1,y1,x2,y2,
      pig_id, behavior, hidden,
      group_id, order,
      category_name
    """
    rows = []

    for cpath in json_paths:
        if not cpath.exists():
            print(f"[WARN] COCO not found: {cpath}")
            continue

        with cpath.open("r", encoding="utf-8") as f:
            data = json.load(f)

        images = data.get("images", [])
        annos  = data.get("annotations", [])
        cats   = data.get("categories", [])

        # map cat_id -> name
        cat_map = {c["id"]: c.get("name", "") for c in cats}

        # map image_id -> (file_name, W, H)
        img_map = {}
        for im in images:
            img_map[im["id"]] = {
                "img_name": Path(im["file_name"]).name,
                "width": im.get("width", None),
                "height": im.get("height", None),
            }

        def parse_attributes(attr):
            pig_id = None
            beh = None
            hidden = None

            if isinstance(attr, dict):
                pig_id = attr.get("ID", None)
                beh = attr.get("Behavior", None)
                hidden = attr.get("Hidden", None)
            elif isinstance(attr, list):
                for a in attr:
                    name = a.get("name")
                    val  = a.get("value")
                    if name == "ID":
                        pig_id = val
                    elif name == "Behavior":
                        beh = val
                    elif name == "Hidden":
                        hidden = val
            return pig_id, beh, hidden

        for an in annos:
            img_info = img_map.get(an["image_id"])
            if img_info is None:
                continue

            img_name = img_info["img_name"]
            W = img_info["width"]
            H = img_info["height"]

            bbox = an.get("bbox", None)
            if not bbox or len(bbox) != 4:
                continue
            x, y, w, h = bbox
            x1, y1 = float(x), float(y)
            x2, y2 = x1 + float(w), y1 + float(h)

            pig_id, beh, hidden = parse_attributes(an.get("attributes", {}))
            group_id, order = parse_burst_from_filename(img_name)

            cat_id = an.get("category_id", None)
            cat_name = cat_map.get(cat_id, "")

            if pig_category_name and cat_name.lower() != pig_category_name.lower():
                continue

            rows.append({
                "img_name": img_name,
                "width": W,
                "height": H,
                "x1": x1, "y1": y1, "x2": x2, "y2": y2,
                "pig_id": pig_id,
                "behavior": beh,
                "hidden": hidden,
                "group_id": group_id,
                "order": order,
                "category_name": cat_name,
            })

        print(f"[LOAD] {cpath}: {len(annos)} annos -> {len(rows)} rows (cumulative)")

    df = pd.DataFrame(rows)
    print(f"[MERGE] total boxes (raw): {len(df)}, images: {df['img_name'].nunique()}")

    before = len(df)
    df = df.drop_duplicates(
        subset=["img_name", "pig_id", "behavior", "x1", "y1", "x2", "y2"],
        keep="first"
    ).reset_index(drop=True)
    after = len(df)
    if after < before:
        print(f"[DEDUP] dropped {before - after} duplicate boxes after merge")

    return df

df_raw = load_and_merge_coco_with_burst(COCO_JSON_PATHS)


In [ ]:
def majority_fix_in_burst(df: pd.DataFrame) -> pd.DataFrame:
    """
    """
    df = df.copy()
    df["pig_id"]   = df["pig_id"].astype(str)
    df["behavior"] = df["behavior"].astype(str)

    priority = {b: i for i, b in enumerate(BEHAVIORS)}

    def _majority_for_group(sub):
        counts = sub["behavior"].value_counts()
        if len(counts) == 0:
            return None
        max_count = counts.max()
        cands = counts[counts == max_count].index.tolist()
        if len(cands) == 1:
            return cands[0]
        cands_sorted = sorted(cands, key=lambda b: priority.get(b, 999))
        return cands_sorted[0]

    majority_map = {}
    mixed_groups = 0

    grp = df.groupby(["group_id", "pig_id"], dropna=False)
    for key, sub in grp:
        uniq_beh = sub["behavior"].dropna().unique()
        if len(uniq_beh) > 1:
            mixed_groups += 1
        majority_map[key] = _majority_for_group(sub)

    print(f"[MAJ] groups with mixed behaviors: {mixed_groups}/{len(majority_map)}")

    def _apply(row):
        key = (row["group_id"], str(row["pig_id"]))
        return majority_map.get(key, row["behavior"])

    df["behavior"] = df.apply(_apply, axis=1)
    return df


In [ ]:
def clean_merged_annotations(df_raw: pd.DataFrame) -> pd.DataFrame:
    df = df_raw.copy()

    before = len(df)
    df = df[df["behavior"].isin(BEHAVIOR_SET)].copy()
    after = len(df)
    pass

    if DROP_HIDDEN:
        before = len(df)
        df = df[df["hidden"].astype(str) != "Yes"].copy()
        after = len(df)
        print(f"[FILTER] drop Hidden=='Yes': {before - after} boxes")

    invalid = (df["x2"] <= df["x1"]) | (df["y2"] <= df["y1"])
    if invalid.sum() > 0:
        pass
        df = df[~invalid].copy()

    if "width" in df.columns and "height" in df.columns:
        def _clamp(row):
            if pd.isna(row["width"]) or pd.isna(row["height"]):
                return row
            W = float(row["width"])
            H = float(row["height"])
            x1 = max(0.0, min(float(row["x1"]), W))
            x2 = max(0.0, min(float(row["x2"]), W))
            y1 = max(0.0, min(float(row["y1"]), H))
            y2 = max(0.0, min(float(row["y2"]), H))
            if x2 <= x1 or y2 <= y1:
                row["drop_bad"] = 1
            row["x1"], row["x2"], row["y1"], row["y2"] = x1, x2, y1, y2
            return row

        df["drop_bad"] = 0
        df = df.apply(_clamp, axis=1)
        bad = df["drop_bad"] == 1
        if bad.sum() > 0:
            print(f"[FILTER] drop {bad.sum()} boxes sau clamp bbox")
            df = df[~bad].copy()
        df = df.drop(columns=["drop_bad"])

    # 4) Majority per (group_id, pig_id)
    df = majority_fix_in_burst(df)

    needed = ["img_name", "group_id", "pig_id", "behavior", "x1", "y1", "x2", "y2"]
    before = len(df)
    df = df.dropna(subset=needed)
    after = len(df)
    pass

    print("===== SUMMARY CLEAN (DataFrame) =====")
    pass
    pass
    pass
    pass
    print(df["behavior"].value_counts())

    df.to_csv(OUT_CLEAN_CSV, index=False, encoding="utf-8")
    print("[SAVE] behavior_clean CSV:", OUT_CLEAN_CSV)

    return df

df_clean = clean_merged_annotations(df_raw)


In [ ]:
def scan_all_images(roots):
    """
    """
    all_names = set()
    name_to_path = {}
    exts = {".jpg", ".jpeg", ".png", ".bmp"}

    for root in roots:
        if not root.exists():
            pass
            continue
        for entry in os.scandir(root):
            if not entry.is_file():
                continue
            ext = Path(entry.name).suffix.lower()
            if ext not in exts:
                continue
            name = entry.name
            all_names.add(name)
            if name not in name_to_path:
                name_to_path[name] = Path(entry.path)

    pass
    return all_names, name_to_path


all_imgs_on_disk, name2path = scan_all_images(IMG_ROOTS)

annotated_imgs = set(df_clean["img_name"].unique())
print(f"[INFO] annotated images (in clean df): {len(annotated_imgs)}")

unannotated_imgs = all_imgs_on_disk - annotated_imgs
print(f"[INFO] unannotated images (no bbox in clean df): {len(unannotated_imgs)}")
pass


In [ ]:
import shutil

copied = 0
for name in annotated_imgs:
    src = name2path.get(name, None)
    if src is None or not src.exists():
        print("[WARN] file missing on disk:", name)
        continue
    dst = IMG_CLEAN_ROOT / name
    if not dst.exists():
        shutil.copy2(src, dst)
        copied += 1

print(f"[COPY] Copied {copied} annotated images to {IMG_CLEAN_ROOT}")


In [ ]:
def build_categories_for_coco():
    return [
        {
            "id": 1,
            "name": "Pig",
            "supercategory": "animal",
        }
    ]


In [ ]:
def build_coco_from_clean_df(df_clean: pd.DataFrame, img_root: Path) -> dict:
    """

      - category_id = 1 ('Pig')
    """
    df = df_clean.copy()

    img_names = sorted(df["img_name"].unique())
    img_id_map = {name: i+1 for i, name in enumerate(img_names)}

    images = []
    for name in img_names:
        sub = df[df["img_name"] == name].iloc[0]
        W = int(sub["width"]) if not pd.isna(sub["width"]) else None
        H = int(sub["height"]) if not pd.isna(sub["height"]) else None

        images.append({
            "id": img_id_map[name],
            "file_name": name,
            "width": W,
            "height": H,
        })

    # annotations
    annotations = []
    ann_id = 1
    for _, row in df.iterrows():
        img_name = row["img_name"]
        image_id = img_id_map[img_name]

        x1, y1, x2, y2 = float(row["x1"]), float(row["y1"]), float(row["x2"]), float(row["y2"])
        w = x2 - x1
        h = y2 - y1
        bbox = [x1, y1, w, h]

        attr = {
            "ID": row["pig_id"],
            "Behavior": row["behavior"],
            "Hidden": row.get("hidden", "No"),
            "group_id": row["group_id"],
            "order": int(row["order"]),
        }

        annotations.append({
            "id": ann_id,
            "image_id": image_id,
            "category_id": 1,  # Pig
            "bbox": bbox,
            "area": float(w * h),
            "iscrowd": 0,
            "attributes": attr,
        })
        ann_id += 1

    coco = {
        "info": {
            "description": "Pig behavior dataset (clean merged)",
            "version": "1.0",
        },
        "licenses": [],
        "categories": build_categories_for_coco(),
        "images": images,
        "annotations": annotations,
    }
    return coco


coco_clean = build_coco_from_clean_df(df_clean, IMG_CLEAN_ROOT)

with OUT_COCO_CLEAN.open("w", encoding="utf-8") as f:
    json.dump(coco_clean, f, ensure_ascii=False, indent=2)

print("[SAVE] COCO clean saved to:", OUT_COCO_CLEAN)
print("images:", len(coco_clean["images"]), "annotations:", len(coco_clean["annotations"]))
